# EcoVision – CNN training notebook
Run `python prepare_dataset.py <raw_folder>` first so that `dataset/train|val|test/<class>/` exists. Run cells top to bottom; the last cells save the model, `class_names.json` and `metrics.json` that the Streamlit app reads.

In [ ]:
import json, numpy as np, pandas as pd, tensorflow as tf, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers, models, callbacks

tf.keras.utils.set_random_seed(42)
ROOT = Path('..') if Path('../dataset').exists() else Path('.')  # works from notebooks/ or project root
DATA, MODEL_DIR, REPORTS = ROOT/'dataset', ROOT/'model', ROOT/'reports'
MODEL_DIR.mkdir(exist_ok=True); REPORTS.mkdir(exist_ok=True)
IMG_SIZE, BATCH, EPOCHS = (128, 128), 32, 30

## 1. Load data (resize happens here)
Three separate folders were created *before* training, so there is no leakage between sets.

In [ ]:
load = lambda split, shuffle: tf.keras.utils.image_dataset_from_directory(
    DATA/split, image_size=IMG_SIZE, batch_size=BATCH, label_mode='int', shuffle=shuffle, seed=42)
train_ds, val_ds, test_ds = load('train', True), load('val', False), load('test', False)
class_names = train_ds.class_names
print(class_names)
y_train = np.concatenate([y.numpy() for _, y in train_ds])
print(pd.Series(y_train).map(dict(enumerate(class_names))).value_counts())

## 2. Data augmentation (training set only)
Rotation, horizontal flip, zoom, width/height shift and brightness change. Validation/test images are never augmented.

In [ ]:
augment = tf.keras.Sequential([
    layers.RandomFlip('horizontal'), layers.RandomRotation(0.08), layers.RandomZoom(0.15),
    layers.RandomTranslation(0.1, 0.1), layers.RandomBrightness(0.2, value_range=(0, 255)),
])
AUTOTUNE = tf.data.AUTOTUNE
train_aug = train_ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds, test_ds = val_ds.prefetch(AUTOTUNE), test_ds.prefetch(AUTOTUNE)

x, _ = next(iter(train_ds)); plt.figure(figsize=(10, 3))
for i in range(5):
    plt.subplot(1, 5, i+1); plt.imshow(augment(x[:1], training=True)[0].numpy().clip(0, 255).astype('uint8')); plt.axis('off')
plt.suptitle('Augmented samples'); plt.show()

## 3. CNN architecture
`Rescaling` performs **normalisation** (0-255 → 0-1) inside the model, so the app and training use identical preprocessing. The last conv layer is named `last_conv` for Grad-CAM.

In [ ]:
inputs = layers.Input(shape=IMG_SIZE + (3,))
x = layers.Rescaling(1/255)(inputs)
x = layers.Conv2D(32, 3, activation='relu', padding='same')(x); x = layers.MaxPooling2D()(x)
x = layers.Conv2D(64, 3, activation='relu', padding='same')(x); x = layers.MaxPooling2D()(x)
x = layers.Conv2D(128, 3, activation='relu', padding='same', name='last_conv')(x); x = layers.MaxPooling2D()(x)
x = layers.Flatten()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)
model = models.Model(inputs, outputs, name='ecovision_cnn')
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 4. Training + validation
Class weights handle imbalance; early stopping restores the best validation weights.

In [ ]:
cw = dict(enumerate(compute_class_weight('balanced', classes=np.arange(len(class_names)), y=y_train)))
cbs = [callbacks.EarlyStopping(patience=6, restore_best_weights=True, monitor='val_loss'),
       callbacks.ReduceLROnPlateau(patience=3, factor=0.5)]
history = model.fit(train_aug, validation_data=val_ds, epochs=EPOCHS, class_weight=cw, callbacks=cbs)

## 5. Accuracy and loss curves

In [ ]:
h = history.history
for key, title in (('accuracy', 'Accuracy'), ('loss', 'Loss')):
    plt.figure(figsize=(6, 4)); plt.plot(h[key], label='Training'); plt.plot(h['val_'+key], label='Validation')
    plt.title(f'Training vs Validation {title}'); plt.xlabel('Epoch'); plt.legend(); plt.grid(alpha=.3)
    plt.savefig(REPORTS/f'{key}_curve.png', dpi=150, bbox_inches='tight'); plt.show()

## 6. Testing: confusion matrix, precision, recall, F1

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred = np.argmax(model.predict(test_ds), axis=1)
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
print(classification_report(y_true, y_pred, target_names=class_names))
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5)); sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix (test set)')
plt.savefig(REPORTS/'confusion_matrix.png', dpi=150, bbox_inches='tight'); plt.show()

## 7. Save model + real metrics (the app reads these)

In [ ]:
model.save(MODEL_DIR/'ecovision_cnn.keras')
(MODEL_DIR/'class_names.json').write_text(json.dumps(class_names))
metrics = dict(train_accuracy=h['accuracy'][-1], val_accuracy=h['val_accuracy'][-1], test_accuracy=float(test_acc),
               train_loss=h['loss'][-1], val_loss=h['val_loss'][-1], test_loss=float(test_loss),
               macro_precision=report['macro avg']['precision'], macro_recall=report['macro avg']['recall'],
               macro_f1=report['macro avg']['f1-score'], per_class={c: report[c] for c in class_names})
(MODEL_DIR/'metrics.json').write_text(json.dumps(metrics, indent=2))
pd.DataFrame(metrics['per_class']).T.round(3)

## 8. Predict on a new image (sanity check)

In [ ]:
from PIL import Image
def predict_file(path):
    img = np.expand_dims(np.asarray(Image.open(path).convert('RGB').resize(IMG_SIZE), dtype='float32'), 0)
    p = model.predict(img, verbose=0)[0]; i = int(p.argmax()); return class_names[i], float(p[i])
# print(predict_file('some_image.jpg'))